[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Transformer_From_Scratch.ipynb)

# Transformer From Scratch — Self-Attention by Hand

**Companion notebook to** [`playbooks/ai/transformers/02_Concepts.md`](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/transformers/02_Concepts.md).

This notebook computes self-attention **using only NumPy** — no PyTorch attention modules, no autograd. Every Q/K/V projection, every dot product, every softmax is computed by hand.

**Why this matters.** When you write `nn.MultiheadAttention(...)` in PyTorch, the framework runs exactly what is in this notebook — just on bigger sequences with more heads. The unique mechanic of transformer architecture is **scaled dot-product attention with multi-head decomposition**. Once that makes sense, the rest follows.

## What you will see
1. Build embeddings, Q/K/V projection matrices, and an attention layer by hand
2. Compute single-head scaled dot-product attention end-to-end
3. Decompose into multi-head attention
4. Verify against PyTorch's `F.scaled_dot_product_attention`
5. Apply causal masking (the decoder's constraint)
6. Add positional encoding
7. Visualize attention patterns

By the end, `nn.MultiheadAttention` and `F.scaled_dot_product_attention` will stop being magic.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__)

## 1. The Setup

3 tokens, embedding dimension `d_model = 4`, multi-head `h = 2` (so `d_k = d_v = 2` per head).

In a real transformer, embeddings come from a learned table; here we use small designed values.

In [ ]:
# 3 tokens, d_model = 4
X = np.array([
    [1.0, 0.0, 1.0, 0.0],   # token 1
    [0.0, 1.0, 0.0, 1.0],   # token 2
    [1.0, 1.0, 0.0, 0.0],   # token 3
], dtype=np.float32)

print(f"Embeddings X shape: {X.shape}  (T=3 tokens, d_model=4)")
print(X)

## 2. The Projection Matrices Q, K, V

In [ ]:
# Q, K, V projection matrices, all (d_model, d_model) = (4, 4)
W_q = np.array([[0.1, 0.2, 0.0, 0.0],
                [0.0, 0.0, 0.3, 0.4],
                [0.5, 0.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 0.5]], dtype=np.float32)

W_k = np.array([[0.0, 0.3, 0.0, 0.2],
                [0.4, 0.0, 0.1, 0.0],
                [0.0, 0.2, 0.5, 0.0],
                [0.3, 0.0, 0.0, 0.4]], dtype=np.float32)

W_v = np.array([[0.5, 0.0, 0.0, 0.0],
                [0.0, 0.5, 0.0, 0.0],
                [0.0, 0.0, 0.5, 0.0],
                [0.0, 0.0, 0.0, 0.5]], dtype=np.float32)

# Output projection
W_o = np.eye(4, dtype=np.float32) * 0.5

print("W_q (4x4):")
print(W_q)

## 3. Project to Q, K, V

In [ ]:
Q = X @ W_q    # (3, 4)
K = X @ W_k    # (3, 4)
V = X @ W_v    # (3, 4)

print("Q:")
print(Q)
print("K:")
print(K)
print("V:")
print(V)

## 4. Single-Head Scaled Dot-Product Attention

The equation:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

### 4.1 Compute Attention Scores

In [ ]:
scores = Q @ K.T    # shape (3, 3)
print("Scores Q · Kᵀ:")
print(scores)
print()
print("Each row is one token's similarity to every token's key.")
print("Row 1 (token 1's query): max value at column 2 → token 1 attends most to token 2")

### 4.2 Scale by sqrt(d_k)

In [ ]:
d_k = 4   # for single-head, d_k = d_model
scaled_scores = scores / np.sqrt(d_k)
print(f"Scaled scores (divided by sqrt({d_k}) = {np.sqrt(d_k):.2f}):")
print(scaled_scores)

**Why scale by sqrt(d_k)?** Without scaling, dot products of high-dimensional vectors grow large. After softmax, large inputs become near-1 (one position) and near-0 (others), pushing softmax into saturation and killing the gradient. Scaling keeps the input distribution well-behaved.

### 4.3 Softmax (Row-wise)

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # subtract max for numerical stability
    ex = np.exp(x)
    return ex / ex.sum(axis=axis, keepdims=True)

A = softmax(scaled_scores, axis=-1)
print("Attention weights A (each row sums to 1):")
print(A)
print()
print(f"Row sums: {A.sum(axis=1)}  (should all be 1.0)")

### 4.4 Weighted Sum

In [ ]:
out_single = A @ V
print("Output (A · V):")
print(out_single)
print()
print("Each row is one token's updated representation —")
print("a weighted combination of all values, with weights from attention.")

## 5. Verify Against PyTorch's `F.scaled_dot_product_attention`

PyTorch has a built-in optimized version (uses FlashAttention under the hood when available).

In [ ]:
import torch
import torch.nn.functional as F

Q_t = torch.tensor(Q).unsqueeze(0)   # (1, T, d_model)
K_t = torch.tensor(K).unsqueeze(0)
V_t = torch.tensor(V).unsqueeze(0)

# PyTorch's scaled_dot_product_attention
out_torch = F.scaled_dot_product_attention(Q_t, K_t, V_t, attn_mask=None, is_causal=False)
out_torch_np = out_torch.squeeze(0).numpy()

print("PyTorch output:")
print(out_torch_np)
print("Manual output:")
print(out_single)
print(f"Match: {np.allclose(out_torch_np, out_single)}")

**Identical numbers.** PyTorch's optimized attention produces exactly the same result as our manual NumPy code — just much faster on long sequences.

## 6. Multi-Head Attention

Single-head attention has one fixed mixing scheme. **Multi-head** runs multiple attention computations in parallel, each with its own slice of the embedding dimensions.

For our setup: `d_model = 4`, `h = 2` heads, so `d_k = d_v = 2` per head.

In [ ]:
H = 2          # number of heads
d_k_h = 2      # dimension per head

# Reshape Q, K, V from (T, d_model) to (T, h, d_k) and transpose to (h, T, d_k)
def split_heads(x, h, d_k_h):
    T = x.shape[0]
    return x.reshape(T, h, d_k_h).transpose(1, 0, 2)  # (h, T, d_k)

Q_h = split_heads(Q, H, d_k_h)
K_h = split_heads(K, H, d_k_h)
V_h = split_heads(V, H, d_k_h)

print(f"Q_h shape: {Q_h.shape}  (h={H} heads, T=3 tokens, d_k={d_k_h})")
print()
print("Head 1 Q:")
print(Q_h[0])
print("Head 2 Q:")
print(Q_h[1])

### 6.1 Compute Attention Per Head

In [ ]:
head_outputs = []
attn_weights_per_head = []

for i in range(H):
    s = Q_h[i] @ K_h[i].T / np.sqrt(d_k_h)
    a = softmax(s, axis=-1)
    o = a @ V_h[i]

    head_outputs.append(o)
    attn_weights_per_head.append(a)

    print(f"\nHead {i+1}:")
    print(f"  scores: {s}")
    print(f"  attention weights:")
    print(f"  {a}")
    print(f"  output:")
    print(f"  {o}")

### 6.2 Concatenate Heads

In [ ]:
# Concatenate along the head dimension and reshape back to (T, d_model)
multi_out = np.concatenate(head_outputs, axis=-1)
print(f"Concatenated multi-head output shape: {multi_out.shape}")
print(multi_out)

### 6.3 Output Projection

In [ ]:
final = multi_out @ W_o
print("After output projection W_o:")
print(final)

## 7. Causal Masking — The Decoder's Constraint

For autoregressive generation, the model must NOT see future tokens. Causal masking zeros out attention to future positions.

For 3 tokens, the causal mask is:
```
1 0 0   ← token 1 attends to itself only
1 1 0   ← token 2 attends to tokens 1, 2
1 1 1   ← token 3 attends to all
```

In [ ]:
# Causal mask: 1 where attention is allowed, 0 where masked
T = 3
mask = np.tril(np.ones((T, T), dtype=np.float32))
print("Causal mask:")
print(mask)
print()

# Apply by adding -inf to masked positions before softmax
scores = Q @ K.T / np.sqrt(d_k)
masked_scores = np.where(mask == 1, scores, -np.inf)
print("Masked scores (-inf hides future positions):")
print(masked_scores)
print()

A_causal = softmax(masked_scores, axis=-1)
print("Causal attention weights:")
print(A_causal)
print()
print("Row 1: only attends to itself")
print("Row 2: attends to tokens 1, 2")
print("Row 3: attends to all")

## 8. Positional Encoding

Self-attention is **permutation-invariant** by default — it cannot distinguish "cat sat" from "sat cat". We must add positional information.

The original Transformer paper used sinusoidal positional encoding.

In [ ]:
def sinusoidal_positional_encoding(T, d_model):
    """Original Transformer paper's positional encoding."""
    pos = np.arange(T)[:, None]    # (T, 1)
    i = np.arange(d_model)[None, :]  # (1, d_model)
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)
    angle_rads = pos * angle_rates

    pe = np.zeros((T, d_model), dtype=np.float32)
    pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pe[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return pe

# Visualize positional encoding for a longer sequence
T_vis = 50
d_vis = 64
pe_vis = sinusoidal_positional_encoding(T_vis, d_vis)

plt.figure(figsize=(10, 6))
plt.imshow(pe_vis, aspect='auto', cmap='RdBu')
plt.xlabel('Embedding dimension')
plt.ylabel('Position in sequence')
plt.title('Sinusoidal positional encoding (50 positions, d_model=64)')
plt.colorbar()
plt.show()

# Apply to our small example
pe = sinusoidal_positional_encoding(3, 4)
print("\nPositional encoding for our 3 tokens (d_model=4):")
print(pe)

**The pattern.** Each row is one position's encoding. Each column is one dimension's frequency. Different frequencies across dimensions let the network attend to "3 positions ahead" or "5 positions ahead" by combining frequencies.

In practice, positional encoding is **added** to token embeddings before the first attention layer:

```python
embeddings = token_embed(input_ids) + pe
```

Modern alternatives (RoPE, ALiBi) work differently but serve the same purpose.

## 9. Visualize Attention Patterns

Look at how each token attends to others — useful for debugging trained models.

In [ ]:
# For our small 3-token example, plot the attention weights from each head
fig, axes = plt.subplots(1, H, figsize=(8, 4))
for i in range(H):
    ax = axes[i]
    im = ax.imshow(attn_weights_per_head[i], cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(['T1', 'T2', 'T3'])
    ax.set_yticklabels(['T1', 'T2', 'T3'])
    ax.set_xlabel('Key (attended to)')
    ax.set_ylabel('Query (attending from)')
    ax.set_title(f'Head {i+1}')

    # Annotate values
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f'{attn_weights_per_head[i][r, c]:.2f}',
                    ha='center', va='center',
                    color='white' if attn_weights_per_head[i][r, c] > 0.5 else 'black')

plt.tight_layout()
plt.show()

**In a trained transformer**, you would see specialized patterns: one head attending to syntactic neighbors, another to long-range dependencies, etc. With our untrained random-ish weights, the patterns are flat.

## What you just did

| Step | What you proved |
|---|---|
| Project to Q, K, V | Three linear projections of the same input vector — that's all queries/keys/values are |
| Scaled dot-product | Each token's query dot-products with every other key, scaled by sqrt(d_k) |
| Softmax | Convert similarities to a probability distribution that sums to 1 |
| Weighted sum | Each token's output is a weighted average of values, weighted by attention |
| PyTorch verification | Manual NumPy attention exactly matches `F.scaled_dot_product_attention` |
| Multi-head | Split d_model into h heads, run attention in parallel, concatenate, project |
| Causal masking | Add -inf to future positions before softmax → autoregressive decoder |
| Positional encoding | Sinusoidal patterns added to embeddings to break permutation invariance |
| Attention visualization | The weights matrix shows what each token attends to — key debugging tool |

When you write `nn.MultiheadAttention` or `F.scaled_dot_product_attention` in PyTorch, you now know exactly what is happening underneath. **Self-attention is dot product + softmax + weighted sum** — three operations, one matrix multiplication each.

The transformer is then just: stack many of these blocks, add residual connections, layer norm, and feedforward layers. The architectural innovation is small; the consequences are immense.

---

## What's Next

You have built attention by hand. Now build the full transformer:

- **[Transformers → 03 Hello World](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/transformers/03_Hello_World.md)** — Complete transformer language model in 80 lines of PyTorch
- **[Transformers → 02 Concepts](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/transformers/02_Concepts.md)** — Encoder/decoder blocks, three architectural variants
- **[Transformers → 04 How It Works](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/transformers/04_How_It_Works.md)** — Pre-LN vs Post-LN, learning rate warmup, attention patterns
- **[Architectures → transformer.md](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/transformers/architectures/transformer.md)** — Single-doc reference covering everything

The math you just walked through is the foundation. ChatGPT, Claude, GPT-4, and every modern LLM run exactly this attention computation — at scale, with FlashAttention optimization, and stacked 12-120 times deep.